In [ ]:
import sys
from pathlib import Path

# The notebooks are run from notebooks/, so the project root has to be on the path
# before any src.* import.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib

matplotlib.use("Agg")  # headless: the notebooks are executed in CI as well as by hand
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.reporting import configure_fonts

configure_fonts()  # CJK labels in charts render as boxes without this
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


# 02 — Audit procedures and Benford's Law

Two families of test, run over the same population:

1. **Nine independent rule-based procedures** — the controls an auditor can read,
   re-perform and challenge.
2. **Benford's Law** — a population-level analytical procedure whose only useful
   output is a disaggregation.

Both are evaluated against the injected ground truth. That is legitimate for *grading*
and it is disclosed here; neither procedure sees a label when it runs.


## 1. How did each procedure perform?

Precision and recall here are measured **per pattern**: for a given rule the positive class is `anomaly_type == rule_key`. So `recall = 1.000` means every voucher injected with *that* pattern was caught — not that every anomaly was.

In [ ]:
evaluation = pd.read_csv(PROJECT_ROOT / "outputs/reports/rule_evaluation.csv")
evaluation = evaluation.sort_values("precision", ascending=False).reset_index(drop=True)

display(
    evaluation[
        ["rule_label", "flagged", "true_positives", "false_positives", "precision", "recall", "f1"]
    ].style.format(
        {"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"}
    ).background_gradient(subset=["precision", "f1"], cmap="RdYlGn", vmin=0, vmax=1)
)

print(f"alerts raised          {int(evaluation['flagged'].sum()):,}")
print(f"true positives         {int(evaluation['true_positives'].sum()):,}")
print(f"false positives        {int(evaluation['false_positives'].sum()):,}")


## 2. The precision/recall trade-off is a workload decision

Two rules carry most of the noise. Plotting alert volume against precision makes the
resourcing consequence obvious: a low-precision rule does not just add a few rows, it
adds hundreds.

In [ ]:
figure, axis = plt.subplots(figsize=(11, 4.5))
colours = ["#2E7D32" if value >= 0.6 else "#C00000" for value in evaluation["precision"]]
axis.barh(evaluation["rule_label"], evaluation["flagged"], color=colours)
axis.invert_yaxis()
axis.set_xlabel("alerts raised")
axis.set_title("Alert volume by procedure (red = precision below 0.6)")
for position, (flagged, precision) in enumerate(zip(evaluation["flagged"], evaluation["precision"])):
    axis.text(flagged, position, f"  {flagged:,} @ P={precision:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

noisy = evaluation[evaluation["precision"] < 0.6]
print(f"{len(noisy)} procedures account for "
      f"{noisy['flagged'].sum():,} of {evaluation['flagged'].sum():,} alerts "
      f"({noisy['flagged'].sum() / evaluation['flagged'].sum():.1%} of the workload) "
      f"while catching {noisy['true_positives'].sum():,} true positives.")
print()
print("These are scoping procedures. Their output defines a population to look at;")
print("it is not evidence about any individual voucher.")


## 3. Why the large-round-amount rule needs three conditions

A naive `amount % 1000 == 0` is almost meaningless: plenty of legitimate payments are
round because rent, retainers and instalments are round. The rule requires a round
amount **and** an amount above the 95th percentile **and** at least CNY 100,000.

In [ ]:
scored = pd.read_parquet(PROJECT_ROOT / "data/processed/transactions_scored.parquet")

round_amounts = scored["is_round_amount"].astype(bool)
naive = round_amounts
three_condition = scored["large_round_amount_flag"].astype(bool)

print(f"naive amount % 1000 == 0        {int(naive.sum()):,} vouchers "
      f"({naive.mean():.2%} of the ledger)")
print(f"round AND >= P95 AND >= 100k    {int(three_condition.sum()):,} vouchers "
      f"({three_condition.mean():.2%} of the ledger)")
print()
print("How much of each rule's output is actually an injected anomaly:")
truth = scored["anomaly_type"].astype("string")
print(f"  naive           {(truth.eq('large_round_amount') & naive).sum() / max(naive.sum(), 1):.1%}")
print(f"  three-condition {(truth.eq('large_round_amount') & three_condition).sum() / max(three_condition.sum(), 1):.1%}")


## 4. Benford's Law: first digit

MAD is the deciding measure, banded per Nigrini. The chi-square is reported beside it and is deliberately not the deciding measure — with 30,000 observations it rejects distributions whose departures are far too small to matter.

In [ ]:
benford = json.loads((PROJECT_ROOT / "outputs/reports/benford_results.json").read_text())
first = benford["first_digit_test"]
table = pd.DataFrame(benford["first_digit_table"])

print(f"n = {first['n_observations']:,}   MAD = {first['mad']:.6f}   "
      f"chi2 = {first['chi_square']:.3f} (df={first['degrees_of_freedom']}, p={first['p_value']:.4f})")
print(f"conformity: {first['conformity']}")
print()
print(first["interpretation"])
print()
print("CAVEAT:", first["caveat"])

figure, axis = plt.subplots(figsize=(10, 4.5))
positions = np.arange(len(table))
axis.bar(positions - 0.2, table["observed_frequency"], width=0.4, label="observed", color="#1F4E79")
axis.bar(positions + 0.2, table["expected_frequency"], width=0.4, label="Benford expected", color="#C00000")
axis.set_xticks(positions)
axis.set_xticklabels(table["digit"])
axis.set_xlabel("leading digit")
axis.set_ylabel("frequency")
axis.set_title("First-digit distribution vs Benford's Law")
axis.legend()
plt.tight_layout()
plt.show()

display(table.style.format({
    "observed_frequency": "{:.4%}", "expected_frequency": "{:.4%}",
    "deviation": "{:+.4%}", "z_score": "{:+.2f}",
}))


## 5. The two tests disagree, and that is the interesting part

The first-two-digits chi-square is very large while its MAD is negligible. The
per-bucket z-scores explain both: 90 buckets each carry a tiny deviation, and chi-square
accumulates them.

In [ ]:
two = benford["first_two_digits_test"]
two_table = pd.DataFrame(benford["first_two_digits_table"])

print(f"first digit      MAD {first['mad']:.6f}  chi2 {first['chi_square']:8.3f}  df {first['degrees_of_freedom']:3d}  p {first['p_value']:.3g}")
print(f"first two digits MAD {two['mad']:.6f}  chi2 {two['chi_square']:8.3f}  df {two['degrees_of_freedom']:3d}  p {two['p_value']:.3g}")
print()
print(f"both are classified '{two['conformity']}' on MAD.")
print()

flagged = two_table[two_table["significant"]]
print(f"buckets with |z| > 1.96: {len(flagged)} of {len(two_table)}")
print(f"expected by chance at the 5% level: {0.05 * len(two_table):.1f}")
print(f"largest |z|: {two_table['z_score'].abs().max():.2f}")
print()
print("MAD measures the average MAGNITUDE of the departure, which is negligible.")
print("Chi-square asks whether the distribution is EXACTLY Benford, which with this")
print("many observations it never is. The MAD is the measure that maps to the audit")
print("question, and 13 exceedances across 90 buckets is a multiple-testing effect,")
print("not evidence of irregularity.")

display(flagged[["digit", "observed_frequency", "expected_frequency", "deviation", "z_score"]]
        .style.format({"observed_frequency": "{:.4%}", "expected_frequency": "{:.4%}",
                       "deviation": "{:+.4%}", "z_score": "{:+.2f}"}))


## 6. Disaggregation is the actual deliverable

A ledger-wide MAD of 0.002 tells an auditor nothing actionable. The per-account breakdown is where an enquiry starts, because deviations in opposite directions cancel out in the aggregate.

In [ ]:
by_account = pd.DataFrame(benford["by_account"]).sort_values("mad", ascending=False)
by_process = pd.DataFrame(benford["by_process"]).sort_values("mad", ascending=False)

print("Accounts, worst conformity first:")
display(by_account[["account_code", "n_observations", "total_amount", "mad", "conformity"]]
        .head(8).style.format({"total_amount": "{:,.0f}", "mad": "{:.5f}"}))

print("Business processes, worst conformity first:")
display(by_process[["process", "n_observations", "total_amount", "mad", "conformity"]]
        .head(8).style.format({"total_amount": "{:,.0f}", "mad": "{:.5f}"}))

nonconforming = by_account[by_account["conformity"] != "Close conformity"]
print(f"{len(nonconforming)} of {len(by_account)} accounts depart from close conformity.")
print("That is where an auditor would look next - and each one has an innocent")
print("explanation to rule out first (policy-constrained amounts, narrow ranges).")


## 7. What Benford's Law cannot do

This is a deliverable, not a footnote:

- A **conforming** distribution is entirely compatible with fabricated entries.
- A **nonconforming** distribution is usually a legitimately skewed account.
- It operates on a **population** and cannot say "this voucher is odd".

It is used here as a disaggregation tool, which is why the statistical component carries a weight of 0.10 — the joint-smallest in the risk score.

In [ ]:
assert "not a test for fraud" in first["caveat"], "the caveat must survive"
print("caveat preserved in the artefact:", first["caveat"])


## What this establishes

- **The nine procedures raise 2,475 alerts on 2,208 vouchers** — 7.33% of the ledger, a
  population a team can actually work.
- **Two procedures produce 51% of the workload for a small share of the detections.**
  Keeping them as scoping procedures rather than findings would halve the review list.
- **The large-round-amount rule needs three conditions.** The naive version flags 505
  vouchers; adding the size and materiality gates is what makes the output usable.
- **Benford is a non-event, and reporting it as such is the point.** Close conformity at
  MAD 0.00218 tells the auditor to spend their time elsewhere. An analytics function
  that always finds something is one nobody trusts.
- **The chi-square and the MAD disagree, and the MAD wins.** That is a judgement about
  what the audit question is, not a technicality.
